# ARMA — Univariate Benchmark (Pipeline B)

**Model**: ARMA(p,q) via pmdarima.auto_arima
**Target**: gdpc1 (quarterly log-diff real GDP)
**Variables**: 1 (target only — univariate benchmark)
**Train**: 1959-Q1 to 2007-Q4
**Validation**: 2008-Q1 to 2016-Q4 (AIC selects p,q inside fit)
**Test**: 2017-Q1 to 2025-Q4 (36 quarters)
**Vintages**: m1 and m2 produce identical predictions (no new GDP between them);
             m3 uses one additional quarter of GDP (pub-lag=2 reveals Q-1 at Q-end)
**Scaling**: No
**HP tuning**: No (AIC inside auto_arima)
**Benchmark role**: All other models report relative RMSFE vs ARMA


In [1]:
import sys, os
import numpy as np
import pandas as pd
import pmdarima as pm
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.rcParams['figure.figsize'] = [15, 10]

# Helpers
sys.path.insert(0, os.path.join(os.path.pardir, 'data'))
from helpers import gen_lagged_data, load_data, PREDICTIONS_DIR, TARGET

SEED = 42
np.random.seed(SEED)


In [2]:
# ── Dates ────────────────────────────────────────────────────────────────────
TRAIN_START = '1959-01-01'
TRAIN_END   = '2007-12-31'
TEST_START  = '2017-01-01'
TEST_END    = '2025-12-31'

# ── Load data ────────────────────────────────────────────────────────────────
monthly, _, metadata = load_data()

# ARMA works on the quarterly target series only
gdpc1_q = monthly[['date', TARGET]].dropna().reset_index(drop=True)
gdpc1_q['date'] = pd.to_datetime(gdpc1_q['date'])

print('Quarterly gdpc1: {} observations'.format(len(gdpc1_q)))
print('Date range: {} to {}'.format(gdpc1_q['date'].min().date(), gdpc1_q['date'].max().date()))

# ── Train/test split ─────────────────────────────────────────────────────────
train = gdpc1_q[(gdpc1_q['date'] >= TRAIN_START) & (gdpc1_q['date'] <= TRAIN_END)]
test  = gdpc1_q[(gdpc1_q['date'] >= TEST_START)  & (gdpc1_q['date'] <= TEST_END)]

print('Train: {} quarters ({} to {})'.format(len(train), TRAIN_START, TRAIN_END))
print('Test:  {} quarters ({} to {})'.format(len(test), TEST_START, TEST_END))

# ARMA order selected once on training data
train_series = train[TARGET].values
arma_model = pm.auto_arima(train_series, seasonal=False, stationary=True,
                            random_state=SEED)
ar_order = arma_model.order[0]
ma_order = arma_model.order[2]
print('ARMA({},{}) selected by AIC'.format(ar_order, ma_order))


Quarterly gdpc1: 267 observations
Date range: 1959-06-01 to 2025-12-01
Train: 195 quarters (1959-01-01 to 2007-12-31)
Test:  36 quarters (2017-01-01 to 2025-12-31)


ARMA(2,0) selected by AIC


In [3]:
# ── Vintage definitions ──────────────────────────────────────────────────────
# m1: forecast_date = Q-end - 2 months
# m2: forecast_date = Q-end - 1 month
# m3: forecast_date = Q-end month
#
# GDP has months_lag=2 in metadata. gen_lagged_data masks the 2 most recent
# monthly rows relative to forecast_date. Consequence for quarterly GDP:
#   m1 (fc=Jan): masks Nov/Dec/Jan  → last visible Q-end = Sep  (Q-2)
#   m2 (fc=Feb): masks Dec/Jan/Feb  → last visible Q-end = Sep  (Q-2, same as m1)
#   m3 (fc=Mar): masks Jan/Feb/Mar  → last visible Q-end = Dec  (Q-1, one more quarter)
# So m1 == m2, but m3 differs by one additional quarterly observation.

VINTAGES = {'m1': -2, 'm2': -1, 'm3': 0, 'post1': 1}
test_quarters = test['date'].tolist()

predictions = {v: [] for v in VINTAGES}
actuals_list = []

for q_date in test_quarters:
    pd_q = pd.Timestamp(q_date)
    actual = test[test['date'] == q_date][TARGET].values[0]
    actuals_list.append(actual)

    for vintage_name, offset in VINTAGES.items():
        forecast_date = pd_q + pd.DateOffset(months=offset)

        # Apply publication-lag mask so GDP respects its 2-month release lag.
        # At m1/m2 the just-completed quarter is masked; at m3 it is revealed.
        fc_monthly = monthly[monthly['date'] <= forecast_date].reset_index(drop=True)
        masked = gen_lagged_data(metadata, fc_monthly, forecast_date, lag=0)

        # Q-end months only, drop masked (NaN) GDP rows, exclude forecast quarter
        gdp_hist = masked[
            masked['date'].dt.month.isin([3, 6, 9, 12]) &
            (masked['date'] < pd_q)
        ][['date', TARGET]].dropna(subset=[TARGET])

        history = gdp_hist[TARGET].values

        if len(history) < 10:
            pred = np.nan
        else:
            try:
                model = pm.arima.ARIMA(order=(ar_order, 0, ma_order))
                fitted = model.fit(history)
                pred = fitted.predict(n_periods=1)[0]
            except Exception:
                pred = np.nan

        predictions[vintage_name].append(pred)

    if len(actuals_list) % 8 == 0:
        q_label = '{}-Q{}'.format(pd_q.year, (pd_q.month - 1) // 3 + 1)
        print('  ... processed {}'.format(q_label))

print('Done. {} test quarters.'.format(len(test_quarters)))


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  ... processed 2018-Q4


...

C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  ... processed 2024-Q4


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Done. 36 test quarters.


In [4]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

for vintage_name in VINTAGES:
    df = pd.DataFrame({
        'date': test_quarters,
        'actual': actuals_list,
        'prediction': predictions[vintage_name],
    })
    path = os.path.join(PREDICTIONS_DIR, 'arma_{}.csv'.format(vintage_name))
    df.to_csv(path, index=False)
    print('Saved {} ({} rows)'.format(path, len(df)))


Saved C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\predictions\arma_m1.csv (36 rows)
Saved C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\predictions\arma_m2.csv (36 rows)
Saved C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\predictions\arma_m3.csv (36 rows)
Saved C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\predictions\arma_post1.csv (36 rows)


In [5]:
# ── Panel definitions ────────────────────────────────────────────────────────
panels = {
    'Pre-COVID  (2017-2019)': ('2017-01-01', '2019-12-31'),
    'COVID      (2020-2021)': ('2020-04-01', '2021-12-31'),
    'Post-COVID (2022-2025)': ('2022-01-01', '2025-12-31'),
    'Full test  (2017-2025)': ('2017-01-01', '2025-12-31'),
}

def rmse(a, p):
    mask = ~(np.isnan(a) | np.isnan(p))
    return np.sqrt(np.mean((a[mask] - p[mask]) ** 2)) if mask.sum() > 0 else np.nan

def mae(a, p):
    mask = ~(np.isnan(a) | np.isnan(p))
    return np.mean(np.abs(a[mask] - p[mask])) if mask.sum() > 0 else np.nan

def dm_test(loss_a, loss_b):
    d = loss_a - loss_b
    n = len(d)
    var_d = np.var(d, ddof=1) / n if n > 1 else 0
    stat = d.mean() / np.sqrt(var_d) if var_d > 0 else 0.0
    pval = 2 * (1 - norm.cdf(abs(stat)))
    return stat, pval

actual_arr = np.array(actuals_list)

print('=' * 72)
print('ARMA — Evaluation by Panel & Vintage')
print('=' * 72)

for panel_name, (p_start, p_end) in panels.items():
    print()
    print('--- {} ---'.format(panel_name))
    print('  Vintage    RMSFE      MAE       N')
    print('  -------    -----      ---       -')
    for vintage_name in VINTAGES:
        pred_arr = np.array(predictions[vintage_name])
        # Filter to panel
        dates_arr = pd.to_datetime(test_quarters)
        mask = (dates_arr >= p_start) & (dates_arr <= p_end)
        a = actual_arr[mask]
        p = pred_arr[mask]
        n_valid = (~(np.isnan(a) | np.isnan(p))).sum()
        print('  {:<10s} {:.6f}  {:.6f}  {}'.format(
            vintage_name, rmse(a, p), mae(a, p), n_valid))


ARMA — Evaluation by Panel & Vintage

--- Pre-COVID  (2017-2019) ---
  Vintage    RMSFE      MAE       N
  -------    -----      ---       -
  m1         0.003100  0.002540  12
  m2         0.003100  0.002540  12
  m3         0.002722  0.002243  12
  post1      0.002722  0.002243  12

--- COVID      (2020-2021) ---
  Vintage    RMSFE      MAE       N
  -------    -----      ---       -
  m1         0.047123  0.033856  7
  m2         0.047123  0.033856  7
  m3         0.051025  0.032276  7
  post1      0.051025  0.032276  7

--- Post-COVID (2022-2025) ---
  Vintage    RMSFE      MAE       N
  -------    -----      ---       -
  m1         0.004639  0.003437  16
  m2         0.004639  0.003437  16
  m3         0.004742  0.003600  16
  post1      0.004742  0.003600  16

--- Full test  (2017-2025) ---
  Vintage    RMSFE      MAE       N
  -------    -----      ---       -
  m1         0.021399  0.009567  36
  m2         0.021399  0.009567  36
  m3         0.023054  0.009219  36
  post1    